In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from rouge_score import rouge_scorer
# Import Accelerator
from accelerate import Accelerator


MAX_NEW_TOKENS = 10
HIDDEN_DIM = 300
SOS_TOKEN = 0
EOS_TOKEN = 1
PAD_TOKEN = 2
UNK_TOKEN = 3


from collections import Counter

def create_vocabulary(paragraphs, titles_train_raw, min_frequency=0.01):
    word_article_count = Counter()  # Track occurrences of words across different articles

    for paragraph in paragraphs:
        unique_words = set(paragraph.split())
        word_article_count.update(unique_words)

    total_articles = len(paragraphs)  # Total number of articles


    filtered_words = {word: count for word, count in word_article_count.items() if (count / total_articles) >= min_frequency}

    # Create vocabulary with only filtered words
    vocabulary = {"< SOS >": SOS_TOKEN, "<EOS>": EOS_TOKEN, "<PAD>": PAD_TOKEN, "<UNK>": UNK_TOKEN}
    vocabulary.update({word: idx + 4 for idx, word in enumerate(filtered_words)})

    # print(f"Number of words appearing in only one article: {sum(1 for count in filtered_words.values() if count == 1)}")
    print(f"Vocabulary size: {len(vocabulary)}")

    return vocabulary


# Tokenization function
def tokenize_text(texts, vocabulary):
    return [[vocabulary.get(word, UNK_TOKEN) for word in text.split()] or [PAD_TOKEN] for text in texts]

# Collate function for DataLoader
def collate_fn(batch):
    inputs, targets = zip(*batch)
    inputs_padded = pad_sequence([torch.tensor(seq, dtype=torch.long) for seq in inputs], batch_first=True, padding_value=PAD_TOKEN)
    targets_padded = pad_sequence([torch.tensor(seq, dtype=torch.long) for seq in targets], batch_first=True, padding_value=PAD_TOKEN)
    return inputs_padded, targets_padded

# Dataset class
class ParagraphTitleDataset(Dataset):
    def __init__(self, input_sequences, target_sequences):
        self.input_sequences = input_sequences
        self.target_sequences = target_sequences
    def __len__(self):
        return len(self.input_sequences)
    def __getitem__(self, idx):
        return self.input_sequences[idx], self.target_sequences[idx]

# Define Encoder
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.embedding = nn.Embedding(input_size, hidden_size, padding_idx=PAD_TOKEN)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
    def forward(self, input_seq):
        input_seq = torch.clamp(input_seq, min=0, max=self.embedding.num_embeddings - 1)
        embedded = self.dropout(self.embedding(input_seq))
        output, hidden = self.gru(embedded)
        hidden = hidden.view(2, hidden.size(1), -1).permute(1, 0, 2).contiguous().view(hidden.size(1), -1)
        return output, hidden

# Define Decoder with ReLU layer after embedding
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size, padding_idx=PAD_TOKEN)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=-1)
    def forward(self, input_token, hidden):
        input_token = torch.clamp(input_token, min=0, max=self.embedding.num_embeddings - 1)
        embedded = self.embedding(input_token).unsqueeze(1)
        embedded = self.relu(embedded)
        output, hidden = self.gru(embedded, hidden)
        output = self.out(output.squeeze(1))
        return output, hidden

# Define Seq2Seq Model
class Seq2SeqRNN(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2SeqRNN, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.hidden_projection = nn.Linear(encoder.gru.hidden_size * 2, decoder.gru.hidden_size)
    def forward(self, input_seq, target_seq=None):
        batch_size = input_seq.size(0)
        encoder_output, encoder_hidden = self.encoder(input_seq)
        decoder_hidden = self.hidden_projection(encoder_hidden).unsqueeze(0)
        decoder_input = torch.full((batch_size, 1), SOS_TOKEN, device=input_seq.device, dtype=torch.long)
        if self.training and target_seq is not None:
            target_len = target_seq.size(1)
            outputs = torch.zeros(batch_size, target_len, self.decoder.out.out_features, device=input_seq.device)
            for t in range(target_len):
                decoder_output, decoder_hidden = self.decoder(decoder_input[:, -1], decoder_hidden)
                outputs[:, t, :] = decoder_output
                if t < target_len - 1:
                    decoder_input = torch.cat([decoder_input, target_seq[:, t].unsqueeze(1)], dim=1)
            return outputs
        else:
            outputs = []
            for _ in range(MAX_NEW_TOKENS):
                decoder_output, decoder_hidden = self.decoder(decoder_input[:, -1], decoder_hidden)
                outputs.append(decoder_output)
                top_token = decoder_output.argmax(dim=-1)
                decoder_input = torch.cat([decoder_input, top_token.unsqueeze(1)], dim=1)
                if torch.all(top_token == EOS_TOKEN):
                    break
            return torch.stack(outputs, dim=1)

# Modified Training function with Accelerator
def train(model, data_loader, optimizer, criterion, accelerator):
    model.train()
    epoch_loss = 0
    for batch_count, (input_seq, target_seq) in enumerate(data_loader, 1):

        optimizer.zero_grad()
        outputs = model(input_seq, target_seq)
        outputs_flat = outputs.contiguous().view(-1, outputs.size(-1))
        target_flat = target_seq.contiguous().view(-1)

        # Fix vocabulary indexing issue
        max_vocab_index = max(vocabulary.values())
        vocab_size = max_vocab_index + 1
        max_target = target_flat.max().item()
        if max_target >= vocab_size:
            accelerator.print(f"Batch {batch_count}: Target index {max_target} exceeds vocab size {vocab_size}")
            continue

        loss = criterion(outputs_flat, target_flat)

        # Use accelerator for backward pass
        accelerator.backward(loss)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(data_loader)
    accelerator.print(f"Training Loss: {avg_loss:.4f}")
    return avg_loss

def evaluate(model, data_loader, vocabulary, accelerator):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = []

    with torch.no_grad():
        for input_seq, target_seq in data_loader:


            outputs = model(input_seq)  # (batch_size, seq_len, vocab_size)


            predicted_tokens = outputs.argmax(dim=-1)  # (batch_size, seq_len)


            predicted_tokens, target_seq = accelerator.gather_for_metrics((predicted_tokens, target_seq))


            generated_texts = decode_outputs(predicted_tokens, vocabulary)
            reference_texts = decode_outputs(target_seq, vocabulary)


            for i in range(len(reference_texts)):
                scores_list.append(scorer.score(reference_texts[i], generated_texts[i]))

    return scores_list

# Decoding function
def decode_outputs(output_sequences, vocabulary):
    index_to_word = {idx: word for word, idx in vocabulary.items()}
    decoded_texts = []

    for sequence in output_sequences:
        words = []
        for idx in sequence:
            token = idx.item()
            if token == EOS_TOKEN:
                break
            if token != SOS_TOKEN and token != PAD_TOKEN:
                words.append(index_to_word.get(token, "<UNK>"))
        decoded_texts.append(" ".join(words))

    return decoded_texts

# Main execution
if __name__ == "__main__":
    import pandas as pd

    # Initialize accelerator
    accelerator = Accelerator(mixed_precision="fp16")

    # Load data
    paragraphs_train_raw = df2[:-500].Text.tolist()
    titles_train_raw = df_title[:-500].title.tolist()

    # Create vocabulary
    vocabulary = create_vocabulary(paragraphs_train_raw, titles_train_raw, min_frequency=0.01)
    max_vocab_index = max(vocabulary.values())
    vocab_size = max_vocab_index + 1
    accelerator.print(f"Vocabulary size: {vocab_size}")

    # Tokenize data
    tokenized_paragraphs_train = tokenize_text(paragraphs_train_raw, vocabulary)
    tokenized_titles_train = tokenize_text(titles_train_raw, vocabulary)

    # Create dataset and dataloader
    train_dataset = ParagraphTitleDataset(tokenized_paragraphs_train, tokenized_titles_train)
    train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, drop_last=True)

    # Create model components
    encoder = EncoderRNN(vocab_size, HIDDEN_DIM)
    decoder = DecoderRNN(HIDDEN_DIM, vocab_size)
    seq2seq_model = Seq2SeqRNN(encoder, decoder)

    # Create optimizer and loss function
    optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

    # Prepare model, optimizer and dataloader with Accelerator
    seq2seq_model, optimizer, train_data_loader = accelerator.prepare(
        seq2seq_model, optimizer, train_data_loader
    )

    # Train the model
    for epoch in tqdm(range(3)):
        train(seq2seq_model, train_data_loader, optimizer, criterion, accelerator)

    # Wait for all processes to reach this point
    accelerator.wait_for_everyone()

    # Unwrap the model before saving
    unwrapped_model = accelerator.unwrap_model(seq2seq_model)

    if accelerator.is_main_process:
        accelerator.print("Training complete!")


In [ ]:
vocabulary

In [ ]:



def evaluate(model, data_loader, vocabulary, accelerator):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = []
    all_predictions = []
    all_references = []

    # For F1 Score calculation
    true_positives = 0
    false_positives = 0
    false_negatives = 0

    with torch.no_grad():
        for input_seq, target_seq in data_loader:


            outputs = model(input_seq)  # (batch_size, seq_len, vocab_size)
            predicted_tokens = outputs.argmax(dim=-1)  # (batch_size, seq_len)


            predicted_tokens, target_seq = accelerator.gather_for_metrics((predicted_tokens, target_seq))


            generated_texts = decode_outputs(predicted_tokens, vocabulary)
            reference_texts = decode_outputs(target_seq, vocabulary)

            for i in range(len(reference_texts)):
                score = scorer.score(reference_texts[i], generated_texts[i])
                scores_list.append(score)
                all_predictions.append(generated_texts[i])
                all_references.append(reference_texts[i])


                pred_words = set(generated_texts[i].split())
                ref_words = set(reference_texts[i].split())


                true_positives += len(pred_words.intersection(ref_words))
                false_positives += len(pred_words - ref_words)
                false_negatives += len(ref_words - pred_words)




    scores_list = accelerator.gather(scores_list)
    all_predictions = accelerator.gather(all_predictions)
    all_references = accelerator.gather(all_references)

    # Gather F1 components
    true_positives = accelerator.gather(torch.tensor([true_positives], device=accelerator.device)).sum().item()
    false_positives = accelerator.gather(torch.tensor([false_positives], device=accelerator.device)).sum().item()
    false_negatives = accelerator.gather(torch.tensor([false_negatives], device=accelerator.device)).sum().item()

    # Calculate F1 score
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return scores_list, all_predictions, all_references, precision, recall, f1_score

# Testing code with Accelerator

test_paragraphs_raw = df_test_text.Text.tolist()
test_titles_raw = df_test_title.title.tolist()

# Convert test data into tokenized format using the same vocabulary as training
tokenized_paragraphs_test = tokenize_text(test_paragraphs_raw, vocabulary)
tokenized_titles_test = tokenize_text(test_titles_raw, vocabulary)

# Create Test Dataset and DataLoader
test_dataset = ParagraphTitleDataset(tokenized_paragraphs_test, tokenized_titles_test)
test_data_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

# Prepare the test dataloader with Accelerator
test_data_loader = accelerator.prepare(test_data_loader)

# Run evaluation
rouge_scores, predicted_titles, actual_titles, precision, recall, f1_score = evaluate(seq2seq_model, test_data_loader, vocabulary, accelerator)

# Calculate and print average ROUGE scores and F1 score
if accelerator.is_main_process:
    avg_rouge = {
        "rouge1": sum([s['rouge1'].fmeasure for s in rouge_scores]) / len(rouge_scores),
        "rouge2": sum([s['rouge2'].fmeasure for s in rouge_scores]) / len(rouge_scores),
        "rougeL": sum([s['rougeL'].fmeasure for s in rouge_scores]) / len(rouge_scores),
    }

    # Print Average ROUGE Scores and F1 Score
    accelerator.print("Average ROUGE Scores:")
    accelerator.print(f"ROUGE-1: {avg_rouge['rouge1']:.4f}")
    accelerator.print(f"ROUGE-2: {avg_rouge['rouge2']:.4f}")
    accelerator.print(f"ROUGE-L: {avg_rouge['rougeL']:.4f}")
    accelerator.print("\nWord-level F1 Metrics:")
    accelerator.print(f"Precision: {precision:.4f}")
    accelerator.print(f"Recall: {recall:.4f}")
    accelerator.print(f"F1 Score: {f1_score:.4f}")



# B2

In [ ]:


def collate_fn(batch):
    inputs, targets = zip(*batch)

    # Pad input sequences
    inputs_padded = pad_sequence([torch.tensor(seq, dtype=torch.long) for seq in inputs], batch_first=True, padding_value=PAD_TOKEN)

    # Pad target sequences (ensure target sequences are padded the same way)
    targets_padded = pad_sequence([torch.tensor(seq, dtype=torch.long) for seq in targets], batch_first=True, padding_value=PAD_TOKEN)

    return inputs_padded, targets_padded



In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from rouge_score import rouge_scorer
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

def load_glove_embeddings(embedding_path, vocabulary, embedding_dim=300):
    """Loads pre-trained GloVe embeddings and returns an embedding matrix."""
    embeddings_index = {}

    with open(embedding_path, "r", encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = vector

    embedding_matrix = np.random.normal(scale=0.6, size=(len(vocabulary), embedding_dim))
    for word, idx in vocabulary.items():
       if idx < vocab_size:  # Ensure index is within bounds
        if word in embeddings_index:
            embedding_matrix[idx] = embeddings_index[word]

    return torch.tensor(embedding_matrix, dtype=torch.float)


In [ ]:
class HierEncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(HierEncoderRNN, self).__init__()
        # Embedding layer with padding_idx to ignore padding tokens during the embedding lookup
        self.embedding = nn.Embedding(vocab_size, hidden_size, padding_idx=PAD_TOKEN)

        # Word-level GRU to process words within each sentence
        self.word_gru = nn.GRU(hidden_size, hidden_size, batch_first=True, bidirectional=True)

        # Sentence-level GRU to process the representation of sentences
        self.sent_gru = nn.GRU(hidden_size * 2, hidden_size, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_seq):
        # Apply embedding to input sequence
        input_seq = torch.clamp(input_seq, min=0, max=self.embedding.num_embeddings - 1)
        embedded = self.dropout(self.embedding(input_seq))

        # Process words within each sentence using GRU
        word_output, word_hidden = self.word_gru(embedded)

        # Averaging word embeddings per sentence
        sentence_embeddings = word_output.mean(dim=1, keepdim=True)

        # Process sentence embeddings using sentence-level GRU
        sent_output, sent_hidden = self.sent_gru(sentence_embeddings)

        # Format hidden state to match EncoderRNN output:
        # From [num_layers*num_directions, batch_size, hidden_size]
        # To [batch_size, hidden_size*2] (to match your existing processing logic)
        hidden = sent_hidden.view(2, sent_hidden.size(1), -1).permute(1, 0, 2).contiguous().view(sent_hidden.size(1), -1)

        return sent_output, hidden


class Decoder2RNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Decoder2RNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size, padding_idx=PAD_TOKEN)
        self.relu = nn.ReLU()
        self.gru1 = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.gru2 = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input_token, hidden, inference=False):
        # Make sure input_token is properly shaped
        input_token = torch.clamp(input_token, min=0, max=self.embedding.num_embeddings - 1)

        # Create embedding with batch dimension
        embedded = self.embedding(input_token).unsqueeze(1)  # [batch_size, 1, hidden_size]
        embedded = self.relu(embedded)

        # First GRU layer
        output1, hidden1 = self.gru1(embedded, hidden)

        # Second GRU layer
        output2, hidden2 = self.gru2(output1, hidden1)

        # Output processing
        output = self.out(output2.squeeze(1))

        return output, hidden2


In [ ]:
class Seq2SeqRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, embedding_path=None,
                 use_hierarchical=False, use_decoder2=False, use_beam_search=False, beam_width=3):

        super(Seq2SeqRNN, self).__init__()

        # Encoder selection
        self.encoder = HierEncoderRNN(vocab_size, hidden_size) if use_hierarchical else EncoderRNN(vocab_size, hidden_size)

        # Decoder selection
        self.decoder = Decoder2RNN(hidden_size, vocab_size) if use_decoder2 else DecoderRNN(hidden_size, vocab_size)

        # Projection layer to handle bidirectional encoder outputs
        self.hidden_projection = nn.Linear(hidden_size * 2, hidden_size)

        # Beam Search
        self.use_beam_search = use_beam_search
        self.beam_width = beam_width
        self.hidden_size = hidden_size

        # Load GloVe embeddings
        if embedding_path:
            self.load_embeddings(embedding_path)

    def load_embeddings(self, embedding_path):

        embedding_matrix = load_glove_embeddings(embedding_path, vocabulary)
        self.encoder.embedding.weight.data.copy_(embedding_matrix)
        self.encoder.embedding.weight.requires_grad = False  # Freeze embeddings

    def forward(self, input_seq, target_seq=None):
        # Get encoder outputs
        encoder_output, encoder_hidden = self.encoder(input_seq)

        # Project encoder hidden state to decoder dimensions
        # Handle different encoder output dimensions
        if encoder_hidden.dim() == 2:  # [batch_size, hidden_size*2]
            # Project from 600 to 300
            projected_hidden = self.hidden_projection(encoder_hidden)
            # Reshape to [1, batch_size, hidden_size] for GRU
            decoder_hidden = projected_hidden.unsqueeze(0)
        else:  # [num_layers, batch_size, hidden_size*2]
            # Project from 600 to 300 for each layer
            batch_size = encoder_hidden.size(1)
            projected_hidden = self.hidden_projection(encoder_hidden.view(-1, encoder_hidden.size(-1)))
            decoder_hidden = projected_hidden.view(encoder_hidden.size(0), batch_size, -1)

        if self.use_beam_search and target_seq is None:
            return self.beam_search_decoder(input_seq, decoder_hidden)

        # Teacher forcing during training
        batch_size = input_seq.size(0)
        decoder_input = torch.full((batch_size, 1), SOS_TOKEN, device=input_seq.device, dtype=torch.long)
        outputs = []

        for i in range(MAX_NEW_TOKENS):
            decoder_output, decoder_hidden = self.decoder(decoder_input[:, -1], decoder_hidden)
            outputs.append(decoder_output)
            top_token = decoder_output.argmax(dim=1)
            decoder_input = torch.cat([decoder_input, top_token.unsqueeze(1)], dim=1)

            # Teacher forcing
            if target_seq is not None and i < target_seq.shape[1]:
                decoder_input[:, -1] = target_seq[:, i]

            if torch.all(top_token == EOS_TOKEN):
                break

        outputs = torch.stack(outputs).transpose(0, 1)  # Shape: (batch_size, seq_len, vocab_size)

        # Ensure that the target sequence and output match in shape
        if target_seq is not None:
            max_len = min(target_seq.size(1), outputs.size(1))
            outputs = outputs[:, :max_len, :]  # Truncate if necessary to match target sequence length

        return outputs

    def beam_search_decoder(self, input_seq, decoder_hidden):

        batch_size = input_seq.size(0)
        all_results = []
        device = input_seq.device

        # Process each item in the batch separately
        for b in range(batch_size):
            # Extract the hidden state for this batch item only
            single_hidden = decoder_hidden.clone()
            if single_hidden.dim() == 3:  # [num_layers, batch_size, hidden_size]
                single_hidden = single_hidden[:, b:b+1, :]
            else:  # [batch_size, hidden_size]
                single_hidden = single_hidden[b:b+1, :].unsqueeze(0)

            beams = [(0.0, [SOS_TOKEN], single_hidden)]  # (score, sequence, hidden_state)

            for _ in range(MAX_NEW_TOKENS):
                new_beams = []
                for score, seq, hidden in beams:
                    if seq[-1] == EOS_TOKEN:
                        new_beams.append((score, seq, hidden))
                        continue

                    # Decoder input for the next token in the sequence
                    decoder_input = torch.tensor([seq[-1]], device=device)

                    # Pass through decoder
                    try:
                        decoder_output, new_hidden = self.decoder(decoder_input, hidden, inference=True)
                    except TypeError:
                        decoder_output, new_hidden = self.decoder(decoder_input, hidden)

                    # Get the top-k probabilities and indices
                    topk_probs, topk_indices = torch.topk(decoder_output, self.beam_width)

                    # Generate new beams with new sequences
                    for i in range(self.beam_width):
                        new_seq = seq + [topk_indices[0][i].item()]
                        new_score = score + topk_probs[0][i].item()
                        new_beams.append((new_score, new_seq, new_hidden))

                # Sort beams by score and keep the top-k
                beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:self.beam_width]

                # Stop if all top beams end with EOS_TOKEN
                if all(seq[-1] == EOS_TOKEN for _, seq, _ in beams[:min(3, len(beams))]):
                    break

            # Get best sequence for this batch item
            best_seq = beams[0][1]
            all_results.append(best_seq)

        # Pad sequences to the same length
        max_length = max(len(seq) for seq in all_results)
        padded_results = [seq + [PAD_TOKEN] * (max_length - len(seq)) for seq in all_results]

        # Convert to tensor
        return torch.tensor(padded_results, device=device)

In [ ]:
import time
from sklearn.metrics import f1_score


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1st train

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from accelerate import Accelerator
import time

# Initialize accelerator
accelerator = Accelerator(mixed_precision="fp16")



paragraphs_train_raw = df2.Text.tolist()  # Load training data
titles_train_raw = df_title.title.tolist()  # Load corresponding titles

paragraphs_test_raw = df_test_text.Text.tolist()  # Load test data
titles_test_raw = df_test_title.title.tolist()  # Load corresponding test titles

vocabulary = create_vocabulary(paragraphs_train_raw,titles_train_raw, min_frequency=0.01)
vocab_size = len(vocabulary)

# Tokenizing Data
tokenized_paragraphs_train = tokenize_text(paragraphs_train_raw, vocabulary)
tokenized_titles_train = tokenize_text(titles_train_raw, vocabulary)

tokenized_paragraphs_test = tokenize_text(paragraphs_test_raw, vocabulary)
tokenized_titles_test = tokenize_text(titles_test_raw, vocabulary)

# Create Datasets & DataLoaders
train_dataset = ParagraphTitleDataset(tokenized_paragraphs_train, tokenized_titles_train)
test_dataset = ParagraphTitleDataset(tokenized_paragraphs_test, tokenized_titles_test)

# train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, drop_last=True)

test_data_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)


seq2seq_model = Seq2SeqRNN(
    vocab_size=vocab_size,
    hidden_size=HIDDEN_DIM,
    embedding_path="/kaggle/input/glove-data/glove.6B.300d.txt",  # Provide GloVe path if available
    use_hierarchical=False,  # Use hierarchical encoder
    use_decoder2=False,  # Use dual-GRU decoder
    use_beam_search=False,  # Enable beam search
    beam_width=5  # Set beam width
)

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)  # Ignore padding during loss calculation

# Prepare model, optimizer and dataloaders with Accelerator
seq2seq_model, optimizer, train_data_loader, test_data_loader = accelerator.prepare(
    seq2seq_model, optimizer, train_data_loader, test_data_loader
)


def train(model, data_loader, optimizer, criterion, num_epochs=3):
    model.train()
    start_time = time.time()
    for epoch in range(num_epochs):
        epoch_start = time.time()
        total_loss = 0
        for input_seq, target_seq in data_loader:


            # Ensure the target sequence and output match in length
            max_len = max(input_seq.size(1), target_seq.size(1))
            target_seq = target_seq[:, :max_len]  # Trim target sequence if necessary

            optimizer.zero_grad()

            # Model forward pass
            outputs = model(input_seq, target_seq)

            # Ensure the output and target are the same shape
            outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten the output for loss calculation
            target_seq = target_seq.view(-1)

            # Apply the CrossEntropy loss while ignoring padding
            loss = criterion(outputs, target_seq)

            # Use accelerator for backward pass
            accelerator.backward(loss)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        epoch_time = time.time() - epoch_start
        accelerator.print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s")

    total_time = time.time() - start_time
    accelerator.print(f"Total training time: {total_time:.2f} seconds")


from rouge_score import rouge_scorer

def evaluate(model, data_loader, vocabulary):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = []
    eval_start = time.time()

    with torch.no_grad():
        for input_seq, target_seq in data_loader:


            outputs = model(input_seq)  # This calls beam_search_decoder if needed

            # Check dimensionality of outputs and handle accordingly
            if outputs.dim() == 1:
                # If outputs is 1D, it's a single sequence (convert to 2D with batch_size=1)
                outputs = outputs.unsqueeze(0)

            # Ensure outputs are the right shape for decoding
            if outputs.dim() == 2:
                # If no vocabulary dimension (just batch x sequence_length), assume these are indices
                # No need for argmax - these are already the best token indices
                generated_indices = outputs
            else:
                # Normal case - outputs are [batch, seq_len, vocab_size]
                generated_indices = outputs.argmax(dim=-1)

            # Gather predictions and targets for metrics calculation
            generated_indices, target_seq = accelerator.gather_for_metrics((generated_indices, target_seq))

            # Slice to match target length if needed
            if target_seq.size(1) < generated_indices.size(1):
                generated_indices = generated_indices[:, :target_seq.size(1)]

            generated_texts = decode_outputs(generated_indices, vocabulary)
            reference_texts = decode_outputs(target_seq, vocabulary)

            # Calculate ROUGE scores
            for i in range(len(reference_texts)):
                scores_list.append(scorer.score(reference_texts[i], generated_texts[i]))

    avg_rouge = {
        "rouge1": sum([s['rouge1'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rouge2": sum([s['rouge2'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rougeL": sum([s['rougeL'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
    }

    all_preds = []
    all_targets = []

    for gen_seq, tgt_seq in zip(generated_indices, target_seq):
        gen_seq = gen_seq.tolist()
        tgt_seq = tgt_seq.tolist()
        gen_seq = [tok for tok in gen_seq if tok != PAD_TOKEN]
        tgt_seq = [tok for tok in tgt_seq if tok != PAD_TOKEN]
        min_len = min(len(gen_seq), len(tgt_seq))
        if min_len > 0:
            all_preds.extend(gen_seq[:min_len])
            all_targets.extend(tgt_seq[:min_len])


    avg_f1 = f1_score(all_targets, all_preds, average='micro') if all_targets else 0.0



    return avg_rouge, avg_f1



accelerator.print(f"Using device: {accelerator.device}")
accelerator.print(f"Mixed precision: {accelerator.mixed_precision}")

accelerator.print("Starting Training...")
train_start = time.time()
train(seq2seq_model, train_data_loader, optimizer, criterion)
train_end = time.time()
accelerator.print(f"Training Completed! Total time: {train_end - train_start:.2f} seconds")

accelerator.wait_for_everyone()


accelerator.print("Evaluating Model...")
eval_start = time.time()
rouge_scores, avg_f1 = evaluate(seq2seq_model, test_data_loader, vocabulary)
eval_end = time.time()
accelerator.print(f"Evaluation Completed! Total time: {eval_end - eval_start:.2f} seconds")
accelerator.print("ROUGE Scores:", rouge_scores)
accelerator.print(f"Average Micro F1 Score: {avg_f1:.4f}")

# Save the model (from main process only)
if accelerator.is_main_process:
    # Unwrap the model before saving
    unwrapped_model = accelerator.unwrap_model(seq2seq_model)
    torch.save(unwrapped_model.state_dict(), "seq2seq_model.pt")
    accelerator.print("Model saved to seq2seq_model.pt")

# 2nd train

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from accelerate import Accelerator
import time

# Initialize accelerator
accelerator = Accelerator(mixed_precision="fp16")



paragraphs_train_raw = df2.Text.tolist()  # Load training data
titles_train_raw = df_title.title.tolist()  # Load corresponding titles

paragraphs_test_raw = df_test_text.Text.tolist()  # Load test data
titles_test_raw = df_test_title.title.tolist()  # Load corresponding test titles


vocabulary = create_vocabulary(paragraphs_train_raw,titles_train_raw, min_frequency=0.01)
vocab_size = len(vocabulary)

# Tokenizing Data
tokenized_paragraphs_train = tokenize_text(paragraphs_train_raw, vocabulary)
tokenized_titles_train = tokenize_text(titles_train_raw, vocabulary)

tokenized_paragraphs_test = tokenize_text(paragraphs_test_raw, vocabulary)
tokenized_titles_test = tokenize_text(titles_test_raw, vocabulary)

# Create Datasets & DataLoaders
train_dataset = ParagraphTitleDataset(tokenized_paragraphs_train, tokenized_titles_train)
test_dataset = ParagraphTitleDataset(tokenized_paragraphs_test, tokenized_titles_test)

# train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, drop_last=True)

test_data_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)


seq2seq_model = Seq2SeqRNN(
    vocab_size=vocab_size,
    hidden_size=HIDDEN_DIM,
    embedding_path="/kaggle/input/glove-data/glove.6B.300d.txt",  # Provide GloVe path if available
    use_hierarchical=True,  # Use hierarchical encoder
    use_decoder2=False,  # Use dual-GRU decoder
    use_beam_search=False,  # Enable beam search
    beam_width=5  # Set beam width
)

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)  # Ignore padding during loss calculation

# Prepare model, optimizer and dataloaders with Accelerator
seq2seq_model, optimizer, train_data_loader, test_data_loader = accelerator.prepare(
    seq2seq_model, optimizer, train_data_loader, test_data_loader
)

def train(model, data_loader, optimizer, criterion, num_epochs=3):
    model.train()
    start_time = time.time()
    for epoch in range(num_epochs):
        epoch_start = time.time()
        total_loss = 0
        for input_seq, target_seq in data_loader:


            # Ensure the target sequence and output match in length
            max_len = max(input_seq.size(1), target_seq.size(1))
            target_seq = target_seq[:, :max_len]  # Trim target sequence if necessary

            optimizer.zero_grad()

            # Model forward pass
            outputs = model(input_seq, target_seq)

            # Ensure the output and target are the same shape
            outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten the output for loss calculation
            target_seq = target_seq.view(-1)

            # Apply the CrossEntropy loss while ignoring padding
            loss = criterion(outputs, target_seq)

            # Use accelerator for backward pass
            accelerator.backward(loss)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        epoch_time = time.time() - epoch_start
        accelerator.print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s")

    total_time = time.time() - start_time
    accelerator.print(f"Total training time: {total_time:.2f} seconds")


from rouge_score import rouge_scorer

def evaluate(model, data_loader, vocabulary):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = []
    eval_start = time.time()

    with torch.no_grad():
        for input_seq, target_seq in data_loader:


            # Get model predictions
            outputs = model(input_seq)  # This calls beam_search_decoder if needed

            # Check dimensionality of outputs and handle accordingly
            if outputs.dim() == 1:
                # If outputs is 1D, it's a single sequence (convert to 2D with batch_size=1)
                outputs = outputs.unsqueeze(0)

            # Ensure outputs are the right shape for decoding
            if outputs.dim() == 2:
                # If no vocabulary dimension (just batch x sequence_length), assume these are indices
                # No need for argmax - these are already the best token indices
                generated_indices = outputs
            else:
                # Normal case - outputs are [batch, seq_len, vocab_size]
                generated_indices = outputs.argmax(dim=-1)

            # Gather predictions and targets for metrics calculation
            generated_indices, target_seq = accelerator.gather_for_metrics((generated_indices, target_seq))

            # Slice to match target length if needed
            if target_seq.size(1) < generated_indices.size(1):
                generated_indices = generated_indices[:, :target_seq.size(1)]

            generated_texts = decode_outputs(generated_indices, vocabulary)
            reference_texts = decode_outputs(target_seq, vocabulary)

            # Calculate ROUGE scores
            for i in range(len(reference_texts)):
                scores_list.append(scorer.score(reference_texts[i], generated_texts[i]))

    avg_rouge = {
        "rouge1": sum([s['rouge1'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rouge2": sum([s['rouge2'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rougeL": sum([s['rougeL'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
    }

    all_preds = []
    all_targets = []

    for gen_seq, tgt_seq in zip(generated_indices, target_seq):
        gen_seq = gen_seq.tolist()
        tgt_seq = tgt_seq.tolist()

        gen_seq = [tok for tok in gen_seq if tok != PAD_TOKEN]
        tgt_seq = [tok for tok in tgt_seq if tok != PAD_TOKEN]
        min_len = min(len(gen_seq), len(tgt_seq))
        if min_len > 0:
            all_preds.extend(gen_seq[:min_len])
            all_targets.extend(tgt_seq[:min_len])


    avg_f1 = f1_score(all_targets, all_preds, average='micro') if all_targets else 0.0



    return avg_rouge, avg_f1



accelerator.print(f"Using device: {accelerator.device}")
accelerator.print(f"Mixed precision: {accelerator.mixed_precision}")

accelerator.print("Starting Training...")
train_start = time.time()
train(seq2seq_model, train_data_loader, optimizer, criterion)
train_end = time.time()
accelerator.print(f"Training Completed! Total time: {train_end - train_start:.2f} seconds")


accelerator.wait_for_everyone()

accelerator.print("Evaluating Model...")
eval_start = time.time()
rouge_scores, avg_f1 = evaluate(seq2seq_model, test_data_loader, vocabulary)
eval_end = time.time()
accelerator.print(f"Evaluation Completed! Total time: {eval_end - eval_start:.2f} seconds")
accelerator.print("ROUGE Scores:", rouge_scores)
accelerator.print(f"Average Micro F1 Score: {avg_f1:.4f}")

# Save the model
if accelerator.is_main_process:
    # Unwrap the model before saving
    unwrapped_model = accelerator.unwrap_model(seq2seq_model)
    torch.save(unwrapped_model.state_dict(), "seq2seq_model.pt")
    accelerator.print("Model saved to seq2seq_model.pt")

# 3rd train

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from accelerate import Accelerator
import time

# Initialize accelerator
accelerator = Accelerator(mixed_precision="fp16")



paragraphs_train_raw = df2.Text.tolist()  # Load training data
titles_train_raw = df_title.title.tolist()  # Load corresponding titles

paragraphs_test_raw = df_test_text.Text.tolist()  # Load test data
titles_test_raw = df_test_title.title.tolist()  # Load corresponding test titles

vocabulary = create_vocabulary(paragraphs_train_raw,titles_train_raw, min_frequency=0.01)
vocab_size = len(vocabulary)

# Tokenizing Data
tokenized_paragraphs_train = tokenize_text(paragraphs_train_raw, vocabulary)
tokenized_titles_train = tokenize_text(titles_train_raw, vocabulary)

tokenized_paragraphs_test = tokenize_text(paragraphs_test_raw, vocabulary)
tokenized_titles_test = tokenize_text(titles_test_raw, vocabulary)

# Create Datasets & DataLoaders
train_dataset = ParagraphTitleDataset(tokenized_paragraphs_train, tokenized_titles_train)
test_dataset = ParagraphTitleDataset(tokenized_paragraphs_test, tokenized_titles_test)

# train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, drop_last=True)

test_data_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

# ====== 3. Initialize Model ======
seq2seq_model = Seq2SeqRNN(
    vocab_size=vocab_size,
    hidden_size=HIDDEN_DIM,
    embedding_path="/kaggle/input/glove-data/glove.6B.300d.txt",  # Provide GloVe path if available
    use_hierarchical=True,  # Use hierarchical encoder
    use_decoder2=True,  # Use dual-GRU decoder
    use_beam_search=False,  # Enable beam search
    beam_width=5  # Set beam width
)

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)  # Ignore padding during loss calculation

# Prepare model, optimizer and dataloaders with Accelerator
seq2seq_model, optimizer, train_data_loader, test_data_loader = accelerator.prepare(
    seq2seq_model, optimizer, train_data_loader, test_data_loader
)


def train(model, data_loader, optimizer, criterion, num_epochs=3):
    model.train()
    start_time = time.time()
    for epoch in range(num_epochs):
        epoch_start = time.time()
        total_loss = 0
        for input_seq, target_seq in data_loader:


            # Ensure the target sequence and output match in length
            max_len = max(input_seq.size(1), target_seq.size(1))
            target_seq = target_seq[:, :max_len]  # Trim target sequence if necessary

            optimizer.zero_grad()

            # Model forward pass
            outputs = model(input_seq, target_seq)

            # Ensure the output and target are the same shape
            outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten the output for loss calculation
            target_seq = target_seq.view(-1)

            # Apply the CrossEntropy loss while ignoring padding
            loss = criterion(outputs, target_seq)

            # Use accelerator for backward pass
            accelerator.backward(loss)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        epoch_time = time.time() - epoch_start
        accelerator.print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s")

    total_time = time.time() - start_time
    accelerator.print(f"Total training time: {total_time:.2f} seconds")

from rouge_score import rouge_scorer

def evaluate(model, data_loader, vocabulary):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = []
    eval_start = time.time()

    with torch.no_grad():
        for input_seq, target_seq in data_loader:


            # Get model predictions
            outputs = model(input_seq)

            # Check dimensionality of outputs and handle accordingly
            if outputs.dim() == 1:
                # If outputs is 1D, it's a single sequence (convert to 2D with batch_size=1)
                outputs = outputs.unsqueeze(0)

            # Ensure outputs are the right shape for decoding
            if outputs.dim() == 2:
                # If no vocabulary dimension (just batch x sequence_length), assume these are indices
                # No need for argmax - these are already the best token indices
                generated_indices = outputs
            else:
                # Normal case - outputs are [batch, seq_len, vocab_size]
                generated_indices = outputs.argmax(dim=-1)

            # Gather predictions and targets for metrics calculation
            generated_indices, target_seq = accelerator.gather_for_metrics((generated_indices, target_seq))

            # Slice to match target length if needed
            if target_seq.size(1) < generated_indices.size(1):
                generated_indices = generated_indices[:, :target_seq.size(1)]

            generated_texts = decode_outputs(generated_indices, vocabulary)
            reference_texts = decode_outputs(target_seq, vocabulary)

            # Calculate ROUGE scores
            for i in range(len(reference_texts)):
                scores_list.append(scorer.score(reference_texts[i], generated_texts[i]))

    avg_rouge = {
        "rouge1": sum([s['rouge1'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rouge2": sum([s['rouge2'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rougeL": sum([s['rougeL'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
    }

    eval_time = time.time() - eval_start
    accelerator.print(f"Evaluation time: {eval_time:.2f} seconds")

    return avg_rouge


accelerator.print(f"Using device: {accelerator.device}")
accelerator.print(f"Mixed precision: {accelerator.mixed_precision}")

accelerator.print("Starting Training...")
train_start = time.time()
train(seq2seq_model, train_data_loader, optimizer, criterion)
train_end = time.time()
accelerator.print(f"Training Completed! Total time: {train_end - train_start:.2f} seconds")

accelerator.wait_for_everyone()


accelerator.print("Evaluating Model...")
eval_start = time.time()
rouge_scores = evaluate(seq2seq_model, test_data_loader, vocabulary)
eval_end = time.time()
accelerator.print(f"Evaluation Completed! Total time: {eval_end - eval_start:.2f} seconds")
accelerator.print("ROUGE Scores:", rouge_scores)

# Save the model
if accelerator.is_main_process:
    # Unwrap the model before saving
    unwrapped_model = accelerator.unwrap_model(seq2seq_model)
    torch.save(unwrapped_model.state_dict(), "seq2seq_model.pt")
    accelerator.print("Model saved to seq2seq_model.pt")

# 4th train

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from accelerate import Accelerator
import time

# Initialize accelerator
accelerator = Accelerator(mixed_precision="fp16")

paragraphs_train_raw = df2.Text.tolist()  # Load training data
titles_train_raw = df_title.title.tolist()  # Load corresponding titles

paragraphs_test_raw = df_test_text.Text.tolist()  # Load test data
titles_test_raw = df_test_title.title.tolist()  # Load corresponding test titles


vocabulary = create_vocabulary(paragraphs_train_raw,titles_train_raw, min_frequency=0.01)
vocab_size = len(vocabulary)

# Tokenizing Data
tokenized_paragraphs_train = tokenize_text(paragraphs_train_raw, vocabulary)
tokenized_titles_train = tokenize_text(titles_train_raw, vocabulary)

tokenized_paragraphs_test = tokenize_text(paragraphs_test_raw, vocabulary)
tokenized_titles_test = tokenize_text(titles_test_raw, vocabulary)

# Create Datasets & DataLoaders
train_dataset = ParagraphTitleDataset(tokenized_paragraphs_train, tokenized_titles_train)
test_dataset = ParagraphTitleDataset(tokenized_paragraphs_test, tokenized_titles_test)

# train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
train_data_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, drop_last=True)

test_data_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

seq2seq_model = Seq2SeqRNN(
    vocab_size=vocab_size,
    hidden_size=HIDDEN_DIM,
    embedding_path="/kaggle/input/glove-data/glove.6B.300d.txt",  # Provide GloVe path if available
    use_hierarchical=True,  # Use hierarchical encoder
    use_decoder2=True,  # Use dual-GRU decoder
    use_beam_search=True,  # Enable beam search
    beam_width=5  # Set beam width
)

optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)  # Ignore padding during loss calculation

# Prepare model, optimizer and dataloaders with Accelerator
seq2seq_model, optimizer, train_data_loader, test_data_loader = accelerator.prepare(
    seq2seq_model, optimizer, train_data_loader, test_data_loader
)

def train(model, data_loader, optimizer, criterion, num_epochs=3):
    model.train()
    start_time = time.time()
    for epoch in range(num_epochs):
        epoch_start = time.time()
        total_loss = 0
        for input_seq, target_seq in data_loader:


            # Ensure the target sequence and output match in length
            max_len = max(input_seq.size(1), target_seq.size(1))
            target_seq = target_seq[:, :max_len]  # Trim target sequence if necessary

            optimizer.zero_grad()

            # Model forward pass
            outputs = model(input_seq, target_seq)

            # Ensure the output and target are the same shape
            outputs = outputs.reshape(-1, outputs.size(-1))  # Flatten the output for loss calculation
            target_seq = target_seq.view(-1)

            # Apply the CrossEntropy loss while ignoring padding
            loss = criterion(outputs, target_seq)

            # Use accelerator for backward pass
            accelerator.backward(loss)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        epoch_time = time.time() - epoch_start
        accelerator.print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}, Time: {epoch_time:.2f}s")

    total_time = time.time() - start_time
    accelerator.print(f"Total training time: {total_time:.2f} seconds")


from rouge_score import rouge_scorer

def evaluate(model, data_loader, vocabulary):
    model.eval()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores_list = []
    eval_start = time.time()

    with torch.no_grad():
        for input_seq, target_seq in data_loader:


            # Get model predictions
            outputs = model(input_seq)

            # Check dimensionality of outputs and handle accordingly
            if outputs.dim() == 1:
                # If outputs is 1D, it's a single sequence (convert to 2D with batch_size=1)
                outputs = outputs.unsqueeze(0)

            # Ensure outputs are the right shape for decoding
            if outputs.dim() == 2:
                # If no vocabulary dimension (just batch x sequence_length), assume these are indices
                # No need for argmax - these are already the best token indices
                generated_indices = outputs
            else:
                # Normal case - outputs are [batch, seq_len, vocab_size]
                generated_indices = outputs.argmax(dim=-1)

            # Gather predictions and targets for metrics calculation
            generated_indices, target_seq = accelerator.gather_for_metrics((generated_indices, target_seq))

            # Slice to match target length if needed
            if target_seq.size(1) < generated_indices.size(1):
                generated_indices = generated_indices[:, :target_seq.size(1)]

            generated_texts = decode_outputs(generated_indices, vocabulary)
            reference_texts = decode_outputs(target_seq, vocabulary)

            # Calculate ROUGE scores
            for i in range(len(reference_texts)):
                scores_list.append(scorer.score(reference_texts[i], generated_texts[i]))

    avg_rouge = {
        "rouge1": sum([s['rouge1'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rouge2": sum([s['rouge2'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
        "rougeL": sum([s['rougeL'].fmeasure for s in scores_list]) / len(scores_list) if scores_list else 0,
    }

    eval_time = time.time() - eval_start
    accelerator.print(f"Evaluation time: {eval_time:.2f} seconds")

    return avg_rouge


accelerator.print(f"Using device: {accelerator.device}")
accelerator.print(f"Mixed precision: {accelerator.mixed_precision}")

accelerator.print("Starting Training...")
train_start = time.time()
train(seq2seq_model, train_data_loader, optimizer, criterion)
train_end = time.time()
accelerator.print(f"Training Completed! Total time: {train_end - train_start:.2f} seconds")

# Wait for all processes to complete training
accelerator.wait_for_everyone()

accelerator.print("Evaluating Model...")
eval_start = time.time()
rouge_scores = evaluate(seq2seq_model, test_data_loader, vocabulary)
eval_end = time.time()
accelerator.print(f"Evaluation Completed! Total time: {eval_end - eval_start:.2f} seconds")
accelerator.print("ROUGE Scores:", rouge_scores)

# Save the model
if accelerator.is_main_process:
    # Unwrap the model before saving
    unwrapped_model = accelerator.unwrap_model(seq2seq_model)
    torch.save(unwrapped_model.state_dict(), "seq2seq_model.pt")
    accelerator.print("Model saved to seq2seq_model.pt")